# Bring in geography

It is time to get your hands on some spatial data. You will not go far
from your `pandas` experience, you’ll just expand it a bit. This section
covers an introduction to `geopandas`, a Python package extending the
capabilities of `pandas` by providing support for geometries,
projections and geospatial file formats. Let’s start with importing
`geopandas`.

In [ ]:
import geopandas as gpd
import shapely

## Datasets

You will be using a few different datasets in this notebook. The first
one contains data on buildings, streets and street junctions of a small
part of Paris from @fleischmann2021Evolution. The data contain some
information on urban morphology derived from these geometries, but
today, you will be mostly interested in geometries, not so much in
attributes.

## Reading files

Assuming you have a file containing both data and geometry
(e.g. GeoPackage, GeoJSON, Shapefile), you can read it using
`geopandas.read_file()`, which automatically detects the file type and
creates a `geopandas.GeoDataFrame`. A `geopandas.GeoDataFrame` is just
like `pandas.DataFrame` but with additional column(s) containing
geometry (in simple terms).

In [ ]:
paris_url = (
    "https://github.com/martinfleis/evolution-gean/raw/main/data/Paris.gpkg"
)
buildings = gpd.read_file(paris_url, layer="buildings")
buildings.head()

> **Explore available layers**
>
> You can quickly check which layers are available in the GPKG file
> using `geopandas.list_layers()`.
>
> ``` python
> gpd.list_layers(paris_url)
> ```

Let’s have a quick look at the `"geometry"` column. This is the special
one enabled by `geopandas`. You can notice that the objects stored in
this column are not `float` or `string` but Polygons of a `geometry`
data type instead. The column is also a `geopandas.GeoSeries` instead of
a `pandas.Series`.

In [ ]:
buildings["geometry"].head()

Polygons are not the only geometry types you can work with. The same
GPKG that contains buildings data also includes street network
geometries of a LineString geometry type.

In [ ]:
street_edges = gpd.read_file(paris_url, layer="edges")
street_edges.head(2)

You can also load another layer, this time with Point geometries
representing street network junctions.

In [ ]:
street_nodes = gpd.read_file(paris_url, layer="nodes")
street_nodes.head(2)

## Writing files

To write a `GeoDataFrame` back to a file, use `GeoDataFrame.to_file()`.
The file is format automatically inferred from the suffix, but you can
specify your own with the `driver=` keyword. When no suffix is given,
GeoPandas expects that you want to create a folder with an ESRI
Shapefile.

In [ ]:
buildings.to_file("buildings.geojson")

## Geometries

Geometries within the *geometry* column are `shapely` objects. GeoPandas
itself is not creating the object but leverages the existing ecosystem
(note that there is a significant overlap of the team writing both
packages to ensure synergy). A typical `GeoDataFrame` contains a single
geometry column, as you know from traditional GIS software. If you read
it from a file, it will most likely be called `"geometry"`, but that is
not always the case. Furthermore, a `GeoDataFrame` can contain multiple
geometry columns (e.g., one with polygons, another with their centroids
and another with bounding boxes), of which one is considered *active*.
You can always get this active column, no matter its name, by using
`.geometry` property.

In [ ]:
buildings.geometry.head()

> **Property vs indexing**
>
> In data frames, you can usually access a column via indexer
> (`df["column_name"]`) or a property (`df.column_name`). However, the
> property is not available when there is either a method (e.g. `.plot`)
> or a built-in property (e.g. `.crs` or `.geometry`) overriding this
> option.

You can quickly check that the geometry is a data type indeed coming
from `shapely`. You will use `shapely` directly in some occasions but in
most cases, any interaction with `shapely` will be handled by
`geopandas`.

In [ ]:
type(buildings.geometry.loc[0])

There is also a handy SVG representation if you are in a Jupyter
Notebook.

In [ ]:
buildings.geometry.loc[0]

If you’d rather see a text representation, you can retrieve a Well-Known
Text using `.wkt` property.

In [ ]:
buildings.geometry.loc[0].wkt

## Projections

But without an assigned coordinate reference system (CRS), you don’t
know where this shape lies on Earth. Therefore, each geometry column has
(optionally) assigned a CRS. That is always available via `.crs`.

In [ ]:
buildings.crs

If you check the type, you’ll notice it comes from the `pyproj` package.
Note that you will likely never interact with that yourself.

In [ ]:
type(buildings.crs)

The object with the CRS information has plenty of useful tricks. You
can, for example, quickly check if it is a geographic (coordinates are
latitude and longitude in degrees) or a projected CRS (x and y in
meters, feet or similar).

In [ ]:
buildings.crs.is_geographic

`geopandas` is like a glue that brings different pieces of the Python
ecosystem together. `pandas` for tabular data, `shapely` for geometries,
`pyogrio` for interaction with GIS file formats or `pyproj` for CRS
management.

## Simple accessors and methods

Now you have a `GeoDataFrame` and can start working with its geometry.

Since there was only one geometry column in the buildings dataset, this
column automatically becomes the *active* geometry and spatial methods
used on the `GeoDataFrame` will be applied to the `"geometry"` column.

### Measuring area

To measure the area of each polygon, access the `GeoDataFrame.area`
attribute, which returns a `pandas.Series`. Note that
`GeoDataFrame.area` is just `GeoSeries.area` applied to the *active*
geometry column.

In [ ]:
buildings["area"] = buildings.area
buildings["area"].head()

### Getting polygon boundary and centroid

`geopandas` allows you a quick manipulation of geometries. For example,
to get the boundary of each polygon (of a LineString geometry type),
access the `GeoDataFrame.boundary` property:

In [ ]:
buildings["boundary"] = buildings.boundary
buildings["boundary"].head()

Since you have saved the boundary as a new column, you now have two
geometry columns in the same `GeoDataFrame`.

You can also create new geometries, which could be, for example, a
buffered version of the original one (i.e., `GeoDataFrame.buffer(10)` to
buffer by 10 meters if your CRS is in meters) or its centroid:

In [ ]:
buildings["centroid"] = buildings.centroid
buildings["centroid"].head()

### Measuring distance

Measuring distance is similarly straightforward. The building data are
from central Paris, so you can try to figure out how far is each of them
from the [Arc de
Triomphe](https://en.wikipedia.org/wiki/Arc_de_Triomphe).

Use the coordinates of the Arc de Triomphe to generate a Point geometry.

In [ ]:
arc = gpd.GeoSeries.from_xy(x=[2.29503], y=[48.87377], crs="EPSG:4326")

> **Use geocoding to get the geometry**
>
> The code above uses known coordinates. If you don’t know them but know
> the address and a name of a place, you can use the built-in geocoding
> capabilities in `geopandas`:
>
> ``` py
> arc = gpd.tools.geocode("Arc de Triomphe, Paris")
> ```

Now you have the Arc de Triomphe as a Point. However, that point is in
latitude and longitude coordinates, which is a different CRS than the
one `buildings` use. They must use the same CRS to measure the distance
between geometries.

In [ ]:
arc = arc.to_crs(buildings.crs)
arc

With a Point based on the correct CRS, you can measure the distance from
each building to the Arc.

In [ ]:
arc_location = arc.geometry.item()
buildings["distance_to_arc"] = buildings.distance(arc_location)
buildings["distance_to_arc"].head()

Using `buildings.distance(arc_location)` measures the distance from
geometries in the active geometry column, which are Polygons in this
case. But you can also measure distance from geometries in any other
column.

In [ ]:
buildings["centroid"].distance(arc_location).head()

Note that `geopandas.GeoDataFrame` is a subclass of `pandas.DataFrame`,
so you have all the `pandas` functionality available to use on the
geospatial dataset — you can even perform data manipulations with the
attributes and geometry information together.

For example, to calculate the average of the distances measured above,
access the `"distance"` column and call the `.mean()` method on it:

In [ ]:
buildings["distance_to_arc"].mean()

Similarly, you can plot the distribution of distances as a histogram.

In [ ]:
_ = buildings["distance_to_arc"].plot.hist(bins=50)

## Making maps

Maps in GeoPandas are of two kinds. Static images and interactive maps
based on [leaflet.js](http://leafletjs.com).

### Static maps

GeoPandas can also plot maps to check how the geometries appear in
space. Call `GeoDataFrame.plot()` to plot the active geometry. To colour
code by another column, pass in that column as the first argument. In
the example below, you can plot the active geometry column and colour
code by the `"distance_to_arc"` column. You may also want to show a
legend (`legend=True`).

In [ ]:
_ = buildings.plot("distance_to_arc", legend=True)

The map is created using the `matplotlib` package. It’s the same that
was used under the hood for all the plots you have done before. You can
use it directly to save the resulting plot to a PNG file.

In [ ]:
import matplotlib.pyplot as plt

If you now create the plot and use the `plt.savefig()` function in the
same cell, a PNG will appear on your disk.

``` py
buildings.plot("distance_to_arc", legend=True)
plt.savefig("distance_to_arc.png", dpi=150)
```

Some plots may not look as nice as the one above. The issue is that
GeoPandas, by default, maps colors linearly between minimum and maximum.
Check how would a plot showing building area look like.

In [ ]:
_ = buildings.plot(buildings.area, legend=True)

This is not very useful. For this reason, GeoPandas allows you to pass a
classification scheme to derive the bins for the colours from the
distribution of values. For example, you can use quantiles.

In [ ]:
_ = buildings.plot(buildings.area, scheme="quantiles", legend=True)

You can see some differentiation on a map now but it is not optimal.
Other schemas might be better for this particular case, like
Jenks-Caspall breaks. You can also specify the number of bins you want.

In [ ]:
_ = buildings.plot(buildings.area, scheme="jenkscaspall", k=10, legend=True)

GeoPandas uses a package called `mapclassify` to do the classification
schemes and you can check its
[documentation](https://pysal.org/mapclassify/api.html) to see which are
available (or create your own).

> **Explaining schemes**
>
> What happens when you apply a classification scheme is that GeoPandas
> derives bins from the distribution of your data using a selected
> algorithm. These are bins using the equal interval, which resembles
> the default linear mapping of colors.
>
> ``` python
> import mapclassify
>
> _ = mapclassify.classify(buildings.area, 'equal_interval').plot_legendgram(
>     inset=False, vlines=True, vlinecolor='red', cmap='viridis'
> )
> ```
>
> Quantiles would look like this.
>
> ``` python
> _ = mapclassify.classify(buildings.area, 'quantiles').plot_legendgram(
>     inset=False, vlines=True, vlinecolor='red', cmap='viridis'
> )
> ```
>
> While Jenks Caspall breaks would do this:
>
> ``` python
> _ = mapclassify.classify(buildings.area, 'jenkscaspall').plot_legendgram(
>     inset=False, vlines=True, vlinecolor='red', cmap='viridis'
> )
> ```
>
> The distribution you are dealing with has a very long tail, so the
> classification scheme needs to adapt.

> **Other resources for static plotting**
>
> Want to know more about static plots? Check [this
> chapter](https://darribas.org/gds_course/content/bC/lab_C.html#styling-plots)
> of *A Course on Geographic Data Science* by @darribas_gds_course or
> [the GeoPandas
> documentation](https://geopandas.org/en/stable/docs/user_guide/mapping.html).

### Interactive maps

You can also explore your data interactively using
`GeoDataFrame.explore()`, which behaves in the same way `.plot()` does
but returns an interactive HTML map instead.

In [ ]:
buildings.explore("distance_to_arc", legend=True)

Using the `GeoSeries` of centroids, you can create a similar plot, but
since you access only a single column, it has no values to show.

In [ ]:
buildings["centroid"].explore()

> **Keeping data around**
>
> If you want to use centroids on a map but keep the data around to have
> them available in the tooltip, you can assign it as an active geometry
> and then use `.explore()`.
>
> ``` python
> buildings.set_geometry("centroid").explore()
> ```

You can also layer multiple geometry layers on top of each other. You
just need to use one plot as a map (`m`) object for the others.

In [ ]:
m = buildings.explore(tiles="CartoDB Positron", popup=True, tooltip=False)
street_edges.explore(m=m, color="black")
street_nodes.explore(m=m, color="pink")
arc.explore(m=m, marker_type="marker")
m.save("paris-map.html")
m

## Constructive methods

Now that you know how to plot, let’s get back to other constructive
operation.

### Simplification

Geometry simplification is implemented using the `simplify()` and
`simplify_coverage()` methods. The first one, simplifies each geometry
on its own, while the latter one simplifies their shared boundaries
together, preserving a valid coverage.

Using a pretty drastic simplification to illustrate the behavior, see
how the individual simplification treats the data

In [ ]:
simplified = buildings.simplify(10)
simplified.explore()

The result no longer respects the original relations between individual
buildings. This is different when using `simplify_coverage()`, which
ensures that all the relations are retained.

In [ ]:
simplified_coverage = buildings.simplify_coverage(10)
simplified_coverage.explore()

Note that the method will work only when your data form a coverage and
will struggle with overlapping polygons and similar situations.

## Spatial predicates

Spatial predicates tell you the spatial relation between two geometries.
Are they equal, intersect, overlap, or are they within another?

You have a location of Arc de Triomphe, allowing you to identify which
of the buildings is the Arc. In other words, you can check which of the
building polygons contain the Arc geometry.

In [ ]:
contains_arc = buildings.contains(arc.item())
contains_arc.head()

A predicate check like contains return a boolean array indicating the
outcome of the check for each geometry. See if there is any `True`.

In [ ]:
contains_arc.any()

There is, so you can map it. The array can be passed directly to `plot`,
no need to turn it into a column.

In [ ]:
ax = buildings.plot(contains_arc, legend=True, cmap='coolwarm')

Get a point representing a location of another important building within
the dataset - Grand Palais. As above, you can generate it directly from
coordinates or using geocoding.

Directly:

In [ ]:
palais = gpd.GeoSeries.from_xy(
    x=[2.31222], y=[48.86616], crs="EPSG:4326"
).to_crs(buildings.crs)

> **Use geocoding to get the geometry**
>
> The code above uses known coordinates. If you don’t know them but know
> the address and a name of a place, you can use the built-in geocoding
> capabilities in `geopandas`:
>
> ``` py
> palais = gpd.tools.geocode("Grand Palais, Paris").to_crs(buildings.crs)
> ```

Then create a virtual straight line connecting the Arc de Triomphe and
Grand Palais. Here comes one of those cases when you use `shapely`
directly.

In [ ]:
palais_art = shapely.LineString([palais.geometry.item(), arc.geometry.item()])

Check visually what you have to get a better understanding.

In [ ]:
ax = buildings.plot()
palais.plot(ax=ax, color='red')
arc.plot(ax=ax, color='orange')
gpd.GeoSeries([palais_art]).plot(ax=ax, color='k')

Now you can use another predicate check to figure out which buildings
intersect this line.

In [ ]:
intersects = buildings.intersects(palais_art)
buildings[intersects].plot()

Similarly, you can check all the budldings within a set distance from
your line. Get those within 200 meters.

In [ ]:
within_200 = buildings.dwithin(palais_art, 200)
buildings[within_200].plot()

The same could be done by buffering the line by 200 meters and doing the
intersection check. However, using `dwithin` predicate directly is much
more efficient.

## Overlays

Overlay operations can alter the data based on superimposition of two
spatial layers.

### Clip

The predicate checks above can filter all the buildings within a set
distance. However, if you want to get only the areas strictly within the
distance, you can clip the geometry.

In [ ]:
clipped_200 = buildings.clip(palais_art.buffer(200))
clipped_200.plot()

The method takes all the geometries within the masking GeoSeries and
returns only those portions of original that are contained within the
mask.

### Spatial join

Another overlay operation is spatial join. This does not alter geometry
but attributes by joining attributes from one set of geometries to the
other based on a spatial predicate (by default intersects).

Load the data capturing boundaries of Parisian Arrondissements
(districts).

In [ ]:
arrondissements = gpd.read_file(
    'https://martinfleischmann.net/sds/geographic_data/data/arrondissements.geojson'
)
arrondissements.head(2)

> **Alternative**
>
> Instead of reading the file directly off the web, it is possible to
> download it manually, store it on your computer, and read it locally.
> To do that, you can follow these steps:
>
> 1.  Download the file by right-clicking on [this
>     link](https://martinfleischmann.net/sds/geographic_data/data/arrondissements.geojson)
>     and saving the file
> 2.  Place the file in the same folder as the notebook where you intend
>     to read it
> 3.  Replace the code in the cell above with:
>
> ``` python
> arrondissements = gpd.read_file(
>     "arrondissements.geojson",
> )
> ```

For any spatial overlay, you need to ensure that both layers share the
CRS.

In [ ]:
arrondissements.crs.equals(buildings.crs)

Given that is not the case, re-project.

In [ ]:
arrondissements = arrondissements.to_crs(buildings.crs)

Now you can check how these two GeoDataFrames relate to each other.

In [ ]:
ax = buildings.plot()
arrondissements.boundary.plot(ax=ax, color='red')

Buildings are on the boundary of four arrondissements. Use spatial join
to link the information about the arrondissement to each building
geometry.

In [ ]:
buildings_joined = buildings.sjoin(arrondissements)
buildings_joined.head()

All of the columns from the `arrondissements` GeoDataFrame are now
linked to the `buildings` GeoDataFrame with values derived based on the
intersection of geometries. This allows you to plot the arrondissement
information on building geometry.

In [ ]:
buildings_joined.plot('l_ar', legend=True)

## Combining and exploding

A use case that may come after a join like this is to combine all
individual geometries from each arrondissement to single one, dissolving
the boundaries where they touch. An operation like this can be done
using the `dissolve` method.

In [ ]:
arr_buildings_combined = buildings_joined.dissolve('l_ar')
arr_buildings_combined

As you can see, the buildings are now combined to a single MultiPolygon
per each arrondissement. This may not be optimal, so you can explode
them to individual Polygons.

In [ ]:
arr_buildings = arr_buildings_combined.explode()
arr_buildings.head(2)

The resulting geometry reflects a dissolution (merge) of all buildings
that are connected. This could’ve been done directly as well but that
would not allow us to illustrate dissolve by a column.

In [ ]:
arr_buildings.explore()

> **Additional reading**
>
> - Check the similar introduction by @darribas_gds_course in his part
>   [*Spatial
>   data*](https://darribas.org/gds_course/content/bC/lab_C.html) to
>   learn more about static plots and how to get a background map with
>   those.
> - Have a look at the chapter [*Choropleth
>   Mapping*](https://geographicdata.science/book/notebooks/05_choropleth.html)
>   explaining how to get choropleth maps from the Geographic Data
>   Science with Python by @rey2023geographic.
> - Another great introduction to GeoPandas is available in [*Vector
>   processing*](https://pythongis.org/part2/chapter-06/index.html)
>   chapter of *Introduction to Python for Geographic Data Analysis* by
>   Henrikki Tenkanen, Vuokko Heikinheimo, and David Whipp.
> - If that is not enough, you can also check [*Geocomputation with
>   Python*](https://py.geocompx.org) by Michael Dorman, Anita Graser,
>   Jakub Nowosad, and Robin Lovelace.